## Neteja, normalitzacio, centralitzacio

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

# ==========================================
# 1. CARGA DE DATOS (Rutas relativas a /Data)
# ==========================================
path = "../Data/" # Ajustar según la ubicación de tu notebook
ventas = pd.read_csv(f"{path}Ventas.csv")
productos = pd.read_csv(f"{path}Productos.csv")
clientes = pd.read_csv(f"{path}Clientes.csv")
campanas = pd.read_csv(f"{path}Campañas.csv")
potencial = pd.read_csv(f"{path}Potencial.csv")

# Formatear fechas
ventas['Fecha'] = pd.to_datetime(ventas['Fecha'])
campanas['Fecha inicio'] = pd.to_datetime(campanas['Fecha inicio'])
campanas['Fecha fin'] = pd.to_datetime(campanas['Fecha fin'])

# ==========================================
# 2. CAPA 2: LIMPIEZA DE RUIDO ADMINISTRATIVO
# ==========================================
# Lógica: Pedido y Cancelación el mismo día con mismo valor = ERROR ADMIN
(descartar)
ventas['abs_valor'] = ventas['Valores_H'].abs()
duplicados_admin = ventas.duplicated(subset=['Fecha', 'Id. Cliente', 'Id. Producto', 'abs_valor'], keep=False)
ventas_sin_ruido = ventas[~duplicados_admin].copy()

# ==========================================
# 3. NORMALIZACIÓN DE UNIDAD (MAPEO A FAMILIA)
# ==========================================
# Según la guía: "La unidad de análisis es siempre la familia, nunca el SKU"
df = ventas_sin_ruido.merge(productos[['Id.Prod', 'Familia_H', 'Bloque analítico']],
                            left_on='Id. Producto', right_on='Id.Prod')
df = df.merge(clientes[['Id. Cliente', 'Provincia']], on='Id. Cliente')

# Agregamos ventas por día, cliente y familia
df_daily = df.groupby(['Id. Cliente', 'Provincia', 'Familia_H', 'Bloque analítico', 'Fecha']).agg({
    'Valores_H': 'sum',
    'Unidades': 'sum'
}).reset_index()

# ==========================================
# 4. DETECCIÓN DE PERIODOS PROMOCIONALES
# ==========================================
def marcar_promos(fecha):
    mask = (campanas['Fecha inicio'] <= fecha) & (campanas['Fecha fin']
>= fecha)
    return 1 if mask.any() else 0

df_daily['is_promo_period'] = df_daily['Fecha'].apply(marcar_promos)

# ==========================================
# 5. NORMALIZACIÓN VS POTENCIAL (Ratio Métrica)
# ==========================================
df_daily = df_daily.merge(potencial[['Id.Cliente', 'Familia',
'Potencial']],
                            left_on=['Id. Cliente', 'Familia_H'],
                            right_on=['Id.Cliente', 'Familia'],
                            how='left')

# Calculamos ratio_vs_potential (valor comprado / potencial estimado)
df_daily['ratio_vs_potential'] = np.where(df_daily['Potencial'] > 0,
                                            df_daily['Valores_H'] /
df_daily['Potencial'],
                                            0)

# ==========================================
# 6. IDENTIFICACIÓN DE OUTLIERS (EVENTOS ESPECIALES)
# ==========================================
# Usamos IQR para marcar pedidos extraordinarios (stockaje, fin de año)
def flag_outliers(group):
    if len(group) < 5: return pd.Series(0, index=group.index)
    Q3 = group['Valores_H'].quantile(0.75)
    IQR = Q3 - group['Valores_H'].quantile(0.25)
    limit = Q3 + (1.5 * IQR)
    return (group['Valores_H'] > limit).astype(int)

df_daily['evento_especial'] = df_daily.groupby(['Id. Cliente',
'Familia_H'])['Valores_H'].transform(flag_outliers)

# ==========================================
# 7. SEGMENTACIÓN POR ANTIGÜEDAD (< 6 MESES)
# ==========================================
primer_pedido = df_daily.groupby('Id. Cliente')['Fecha'].transform('min')
df_daily['meses_antiguedad'] = (df_daily['Fecha'] -
primer_pedido).dt.days / 30
df_daily['historico_incompleto'] = (df_daily['meses_antiguedad'] <
6).astype(int)

# ==========================================
# 8. GUARDADO DE RESULTADO
# ==========================================
# Columnas finales necesarias para la Capa 3 (Feature Engineering)
cols_finales = [
    'Fecha', 'Id. Cliente', 'Provincia', 'Familia_H', 'Bloque analítico',
    'Valores_H', 'ratio_vs_potential', 'is_promo_period',
    'evento_especial', 'historico_incompleto'
]

df_final = df_daily[cols_finales].sort_values(['Id. Cliente', 'Fecha'])
df_final.to_csv(f"{path}Dataset_Limpiado_Normalizado.csv", index=False)

print("✅ Proceso completado.")
print(f"Total registros limpios: {len(df_final)}")
print(f"Puntos de señal (eventos especiales) detectados:
{df_final['evento_especial'].sum()}")